In [1]:
from vizdoom import *
import cv2
import numpy as np
import time
import random
from matplotlib import pyplot as plt
import pandas as pd
import re
import pandas as pd
import csv
import os
from tqdm import tqdm

# Gym imports
import gymnasium as gym
from gymnasium import Env
from gymnasium.spaces import Discrete, Box

# Ollama imports
import requests

In [2]:
# Stable Baselines3 imports
from stable_baselines3 import PPO
from stable_baselines3.common import env_checker
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback, CallbackList
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.monitor import Monitor

In [3]:
# Paths
CONFIG_PATH = './github/ViZDoom/scenarios/take_cover.cfg'
LOG_DIR = './logs'

In [4]:
class PerceptionLayer:
    URGENCY_IMMINENT   =  150
    URGENCY_APPROACHING = 400
    SIDE_THRESHOLD = 15
    URGENCY_ENC = {"DISTANT": 0, "APPROACHING": 1, "IMMINENT": 2}
    SIDE_ENC    = {"CENTER": 0, "LEFT": 1, "RIGHT": 2}
    DIR_ENC     = {"PARALLEL TO": 0, "AWAY FROM": 1, "TOWARD": 2}
    HIT_ENC     = {None: 0, "HIT": 1, "FATAL": 2}
    MAX_PROJ    = 3

    ACTION_NAMES = {0: "LEFT", 1: "RIGHT"}

    def __init__(self):
        self.prev_health  = 100
        self.last_action  = None
        self.step_count   = 0

    def reset(self):
        self.prev_health = 100
        self.last_action = None
        self.step_count  = 0

    def update(self, state):
        self.step_count += 1

        player_y, health, hit = self._get_player_state(state)
        projectiles = self._get_projectiles(state, player_y)
        action = self.last_action
        llm_text =  self._serialize(health, projectiles, hit, action)
        df_text = self._serialize_for_llm(health, projectiles, hit, action)

        return llm_text, df_text
    
    def update_action(self, action):
        self.last_action = action
    
    def _get_player_state(self, state):
        vars = state.game_variables
        health = vars[0]
        player_y = vars[2]

        delta_h = health - self.prev_health
        self.prev_health = health
        hit = self._detect_hit(health, delta_h)

        return player_y, health, hit
    
    def _detect_hit(self, health, delta_h):
        if health <= 0:
            return "FATAL"
        if delta_h < 0:
            return "HIT"
        return None
    
    def _get_projectiles(self, state, player_y):
        projectiles = []
        for label in state.labels:
            if label.object_name == "DoomImpBall":
                delta_y = label.object_position_y - player_y
                delta_x = label.object_position_x
                projectiles.append({
                "delta_y": delta_y,
                "delta_x": delta_x,
                "side":    self._side(delta_y),
                "urgency": self._urgency(delta_x),
                })
        
        # Returns projectiles sorted by urgency (closest to player first)
        return sorted(projectiles, key=lambda p: abs(p["delta_x"]))
    
    def _side(self, delta_y: float):
        if abs(delta_y) < self.SIDE_THRESHOLD:
            return "CENTER"
        return "LEFT" if delta_y < 0 else "RIGHT"

    def _urgency(self, delta_x: float):
        dist = abs(delta_x)
        if dist < self.URGENCY_IMMINENT:
            return "IMMINENT"
        elif dist < self.URGENCY_APPROACHING:
            return "APPROACHING"
        return "DISTANT"
    
    def _evaluate_direction(self, action, threat_side):
        if threat_side == "CENTER":
            return "PARALLEL TO"
        if (threat_side == "RIGHT" and action == 1) or (threat_side == "LEFT" and action == 0):
            return "TOWARD"
        else:
            return "AWAY FROM"
    
    def _serialize(self, health, projectiles, hit, action):
        lines = []

        if not projectiles:
            return "No projectiles visible."
        else:
            if hit == "FATAL":
                lines.append("Health: 0/100 (DIED this step)")
            elif hit == "HIT":
                lines.append(f"Health: {int(health)}/100 (first hit, critical condition)")
            else:
                lines.append(f"Health: {int(health)}/100")
                
            lines.append(f"{len(projectiles)} projectile(s) detected:")
            for i, p in enumerate(projectiles, 1):
                if action is not None:
                    direction = self._evaluate_direction(action, p['side'])
                    lines.append(f"  {i}. {p['urgency'].upper()} threat on the {p['side']} side. The player MOVED {self.ACTION_NAMES.get(action)}. The player is MOVING {direction} this threat")
                else:
                    lines.append(f"  {i}. {p['urgency'].upper()} threat on the {p['side']} side")

        return "\n".join(lines)
    
    def _serialize_for_llm(self, health, projectiles, hit, action):
        hit_val = hit if hit else "NONE"
        act_val = self.ACTION_NAMES.get(action, "NONE")
        
        parts = [f"H:{int(health)}", f"HIT:{hit_val}", f"ACT:{act_val}", f"NP:{len(projectiles)}"]
        
        for i, p in enumerate(projectiles):
            dir_val = self._evaluate_direction(action, p['side']) if action is not None else "NONE"
            p_str = f"P{i+1}:{p['urgency'][:4]},{p['side'][:4]},{dir_val[:4]}"
            parts.append(p_str)
            
        return " | ".join(parts)


In [5]:
class LLMAgent:
    def __init__(self, model_name, backend_url):
        self.backend_url = backend_url
        self.model_name = model_name
        self.parse_failures = 0
        self._cache = {}
        self.payload = {
            "model" : self.model_name,
            "system" : self._build_system_prompt(),
            "prompt" : "",
            "stream" : False,
        }

    def evaluate(self, state_text, action_taken):
        if state_text == "No projectiles visible.":
            return 0.0
        
        prompt  = self._build_prompt(state_text)
        response = self._call_llm(prompt)
        reward = self._parse_reward(response)
        
        return reward
    
    def _build_system_prompt(self):
        return """You are a reward model for a dodge game. Evaluate the player's action and return a reward score.
        ### SCORING CRITERIA:
        - Player moving AWAY from IMMINENT threat:    +1.0
        - Player moving TOWARD IMMINENT threat:       -1.0
        - Player moving AWAY from APPROACHING threat: +0.5
        - Player moving TOWARD APPROACHING threat:    -0.5
        - No threats:                                 +0.0

        ### FORMULA:
        TOTAL = mean(threat scores)

        ### STEP BY STEP:
        1. For each projectile, check if player is moving toward or away and assign score
        2. Compute mean of all threat scores

        ### OUTPUT FORMAT:
        Write the final line exactly as: REWARD: <number>"""
    
    def _build_prompt(self, state_text):
        return f"""### NOW EVALUATE: {state_text}"""

    def _call_llm(self, prompt):
        self.payload["prompt"] = prompt

        try:
            response = requests.post(self.backend_url, json= self.payload)

            # Check response status
            if response.status_code == 200:
                return response.json()["response"]
            else:
                print(f"Ollama returned error: {response.status_code}")
                return ""
        except requests.exceptions.ConnectionError:
            print("Failed to connect to Ollama backend.")
            return ""   
    
    def _parse_reward(self, response):
        try:
            match = re.search(r'REWARD:\s*([+-]?\d+\.?\d*)', response)
            if match:
                return max(-5.0, min(5.0, float(match.group(1))))
            self.parse_failures += 1
            return 0.0 # Fallback if no number found
        except ValueError:
            self.parse_failures += 1
            return 0.0 # Fallback 

In [6]:
# Create Vizdoom OpenAI Gym Environment
class VizDoomGym(Env): 
    def __init__(self, render=False): 
        # Setup the game 
        super().__init__()
        self.frame_skip = 4
        self.game = DoomGame()
        self.game.load_config(CONFIG_PATH)
        
        # Render frame logic
        self.game.set_window_visible(render)
        
        # Start the game 
        self.game.init()
        
        # Create the action space and observation space
        self.observation_space = Box(low=0, high=255, shape=(100,160,1), dtype=np.uint8) 
        self.action_space = Discrete(2) # Move left, move right
        self._actions = np.eye(2, dtype=np.uint8)
        
    # This is how we take a step in the environment
    def step(self, action):
        # Specify action and take step 
        reward = self.game.make_action(self._actions[action].tolist(), self.frame_skip) 
        
        # Get the new state of the game and check if it's done
        if self.game.get_state(): 
            obs    = self._process_frame(self.game.get_state().screen_buffer)
            health = self.game.get_state().game_variables[0]
            info   = {"health": health}
        else: 
            obs = np.zeros(self.observation_space.shape, dtype=np.uint8)
            info = {"health": 0}
        
        terminated = self.game.is_episode_finished()
        truncated = False

        return obs, reward, terminated, truncated, info
    
    # Define how to render the game or environment 
    def render(self): 
        pass
    
    # Starting a new game 
    def reset(self, seed=None, options=None): 
        super().reset(seed=seed)
        self.game.new_episode()
        state = self.game.get_state()

        if state is not None:
            obs = self._process_frame(state.screen_buffer)
        else:
            obs = np.zeros(self.observation_space.shape, dtype=np.uint8)

        return obs, {}
    
    # Call to close down the game
    def close(self): 
        self.game.close()

    def _process_frame(self, buffer: np.ndarray):
        hwc  = np.moveaxis(buffer, 0, -1)                           
        gray = cv2.cvtColor(hwc, cv2.COLOR_RGB2GRAY)               
        resized = cv2.resize(gray, (160, 100), interpolation=cv2.INTER_CUBIC)
        clipped = np.clip(resized, 0, 255).astype(np.uint8)         
        return clipped.reshape(100, 160, 1)

### Extracting states from real-world instances

In [7]:
env = VizDoomGym(render=False)
perception = PerceptionLayer()
agent = LLMAgent(model_name="llama3.2:3b", backend_url="http://localhost:11434/api/generate")
episodes = 50

csv_path = os.path.join(LOG_DIR, f"game_states.csv")
# txt_path = os.path.join(LOG_DIR, f"game_states.txt")
csv_columns = ["full_text", "summ_text", "action", "reward_base", "reward_llm"]

#, open(txt_path, "w") as txt_file
with open(csv_path, "w", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=csv_columns)
    writer.writeheader()

    for episode in tqdm(range(episodes)):
        obs, info = env.reset()
        perception.reset()
        truncated  = False
        terminated = False
        step = 0

        while not terminated and not truncated:
            step += 1
            state = env.game.get_state()
            state_text, df_text = perception.update(state)

            action = random.randint(0, 1)
            llm_reward = agent.evaluate(state_text, action)
            obs, reward, terminated, truncated, info = env.step(action)
            perception.update_action(action)

            if state_text != "No projectiles visible.":
                # txt_file.write(f"{'='*60}\n")
                # txt_file.write(f"Episode {episode:04d} | Step {step:04d}\n")
                # txt_file.write(f"{'-'*60}\n")
                # txt_file.write(state_text + "\n")
                # txt_file.write(f"Reward base : {reward:.4f}\n")
                # txt_file.write(f"Reward LLM  : {llm_reward:.4f}\n")
                # txt_file.write("\n")

                row = {
                    "full_text" : state_text,
                    "summ_text" : df_text,
                    "action" : action,
                    "reward_base": reward,
                    "reward_llm":  llm_reward,
                }
                writer.writerow(row)

env.close()

100%|██████████| 50/50 [1:26:00<00:00, 103.21s/it]
